### ==========================================
### Phase 1: Data Loading & Type Identification
### ==========================================

In [1]:
import pandas as pd

In [36]:
df=pd.read_csv("Titanic-Dataset.csv")

numeric_cols=df.select_dtypes(include=['number']).columns.tolist()
categorical_cols=df.select_dtypes(include=['object','category']).columns.tolist()

print(f"Total of columns ({len(df.columns)})")
print(f"Total of numeric columns ({len(numeric_cols)})")
print(f"Total of categorical columns ({len(categorical_cols)})")

Total of columns (12)
Total of numeric columns (7)
Total of categorical columns (5)


### ==========================================
### Phase 2: Engine Logic & Issue Detection
### ==========================================

In [47]:
def filter_continuous_numeric_cols(df, numeric_cols, unique_threshold=10):
    continuous_cols = []
    
    for col in numeric_cols:
        if df[col].nunique() <= unique_threshold:
            continue
        if col_lower == "id" or col_lower.endswith("_id") or col_lower.startswith("id_"):
            continue
        continuous_cols.append(col)
    return continuous_cols

In [38]:
def check_data_health(df, top_n=5):
    total_rows = len(df)
    duplicate_rows = int(df.duplicated().sum())
    duplicate_pct = round((duplicate_rows / total_rows) * 100, 2)
    
    null_counts = df.isnull().sum()
    null_cols = null_counts[null_counts > 0]
    
    missing_summary = {}
    for col, count in null_cols.items():
        missing_summary[col] = {
            "count": int(count),
            "percentage": round((count / total_rows) * 100, 2)
        }
    
    sorted_missing = sorted(missing_summary.items(), key=lambda x: x[1]['count'], reverse=True)[:top_n]
    
    return {
        "duplicates": {
            "count": duplicate_rows,
            "percentage": duplicate_pct
        },
        "total_cols_with_missing": len(missing_summary),
        "top_missing_columns": dict(sorted_missing)
    }

In [39]:
def detect_outliers(df, continuous_numeric_cols, top_n=5):
    outliers_summary = {}
    total_rows = len(df)
    
    for col in continuous_numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        if IQR == 0:
            continue
            
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
        outlier_count = int(outlier_mask.sum())
        
        if outlier_count > 0:
            outliers_summary[col] = {
                "count": outlier_count,
                "percentage": round((outlier_count / total_rows) * 100, 2)
            }
            
    sorted_outliers = sorted(outliers_summary.items(), key=lambda x: x[1]['count'], reverse=True)[:top_n]
    return {
        "total_numeric_cols_with_outliers": len(outliers_summary),
        "top_outlier_columns": dict(sorted_outliers)
    }

In [40]:
def detect_high_cardinality(df, categorical_cols, threshold=0.5):
    total_rows = len(df)
    high_card_cols = {}
    
    for col in categorical_cols:
        unique_count = df[col].nunique()
        ratio = unique_count / total_rows
        
        if ratio >= threshold:
            high_card_cols[col] = {
                "unique_count": unique_count,
                "cardinality_ratio": round(ratio, 2)
            }
    return {
        "high_cardinality_cols_count": len(high_card_cols),
        "columns": high_card_cols
    }

In [41]:
def detect_high_correlation(df, numeric_cols, threshold=0.7, top_n=5):
    if len(numeric_cols) < 2:
        return {"total_high_correlations": 0, "top_correlations": []}
        
    corr_matrix = df[numeric_cols].corr().abs()
    high_corr_list = []
    
    for i in range(len(numeric_cols)):
        for j in range(i + 1, len(numeric_cols)):
            col1 = numeric_cols[i]
            col2 = numeric_cols[j]
            corr_val = corr_matrix.loc[col1, col2]
            
            if pd.notnull(corr_val) and corr_val >= threshold:
                high_corr_list.append({
                    "pair": f"{col1} <-> {col2}",
                    "correlation": round(corr_val, 2)
                })
                
    sorted_corr = sorted(high_corr_list, key=lambda x: x['correlation'], reverse=True)[:top_n]
    return {
        "total_high_correlations": len(high_corr_list),
        "top_correlations": sorted_corr
    }

### ==========================================
### Phase 2: Master Orchestrator (Unified Metadata)
### ==========================================

In [42]:
def detect_issues(df):
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    
    continuous_numeric_cols = filter_continuous_numeric_cols(df, numeric_cols)
    
    return {
        "dataset_summary": {
            "total_rows": len(df),
            "total_cols": len(df.columns),
            "numeric_cols_count": len(numeric_cols),
            "categorical_cols_count": len(categorical_cols)
        },
        "data_health": check_data_health(df),
        "outliers": detect_outliers(df, continuous_numeric_cols),
        "high_cardinality": detect_high_cardinality(df, categorical_cols),
        "high_correlations": detect_high_correlation(df, continuous_numeric_cols)
    }

In [46]:
full_analysis = detect_issues(df)
full_analysis

{'dataset_summary': {'total_rows': 891,
  'total_cols': 12,
  'numeric_cols_count': 7,
  'categorical_cols_count': 5},
 'data_health': {'duplicates': {'count': 0, 'percentage': 0.0},
  'total_cols_with_missing': 3,
  'top_missing_columns': {'Cabin': {'count': 687, 'percentage': 77.1},
   'Age': {'count': 177, 'percentage': 19.87},
   'Embarked': {'count': 2, 'percentage': 0.22}}},
 'outliers': {'total_numeric_cols_with_outliers': 2,
  'top_outlier_columns': {'Fare': {'count': 116, 'percentage': 13.02},
   'Age': {'count': 11, 'percentage': 1.23}}},
 'high_cardinality': {'high_cardinality_cols_count': 2,
  'columns': {'Name': {'unique_count': 891, 'cardinality_ratio': 1.0},
   'Ticket': {'unique_count': 681, 'cardinality_ratio': 0.76}}},
 'high_correlations': {'total_high_correlations': 0, 'top_correlations': []}}